# 02 — Data Cleaning & Exploratory Data Analysis

## Objective

This notebook prepares the raw dataset for machine learning while preserving the original information as much as possible.

The main objectives are:

- Handle unknown and missing values
- Investigate identifier columns
- Inspect repeated patients
- Validate categorical values
- Identify unusable or non-informative features
- Document all cleaning decisions
- Prepare a clean dataset for subsequent modeling

No machine learning model is trained in this phase.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

DATA_PATH = Path("../data/raw/diabetic_data.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (101766, 50)


## 1. Preserve the Raw Dataset

A separate copy is created so that all cleaning operations can be compared against the original raw data.

The raw dataset itself will never be modified.

In [2]:
df_clean = df.copy()

print(f"Raw shape:   {df.shape}")
print(f"Clean shape: {df_clean.shape}")

Raw shape:   (101766, 50)
Clean shape: (101766, 50)


## 2. Unknown Value Representation

The dataset uses `?` to represent unknown or unavailable information.

Before deciding how to handle these values, we quantify their frequency across all features.

In [3]:
unknown_counts = (df_clean == "?").sum()

unknown_summary = (
    unknown_counts[unknown_counts > 0]
    .sort_values(ascending=False)
    .to_frame("unknown_count")
)

unknown_summary["percentage"] = (
    unknown_summary["unknown_count"] / len(df_clean) * 100
).round(2)

unknown_summary

,unknown_count,percentage
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


## 3. Identifier and Patient-Level Structure

The dataset contains both encounter-level and patient-level identifiers.

We inspect:

- `encounter_id`: identifier for each hospital encounter
- `patient_nbr`: identifier for the patient

These columns are not automatically removed at this stage. We first investigate their uniqueness and whether patients have multiple encounters.

In [4]:
print(f"Unique encounter_id: {df_clean['encounter_id'].nunique():,}")
print(f"Total rows:          {len(df_clean):,}")

print(f"\nUnique patient_nbr:   {df_clean['patient_nbr'].nunique():,}")
print(f"Patients with >1 encounter: {(df_clean['patient_nbr'].value_counts() > 1).sum():,}")

Unique encounter_id: 101,766
Total rows:          101,766

Unique patient_nbr:   71,518
Patients with >1 encounter: 16,773


## 4. Repeated Patient Encounters

Multiple encounters from the same patient are present in the dataset.

We quantify how many encounters are associated with each patient to understand the patient-level structure before defining the train/test splitting strategy.

In [5]:
patient_encounters = df_clean["patient_nbr"].value_counts()

patient_summary = patient_encounters.describe()

print(patient_summary)

print("\nEncounter frequency:")
print(patient_encounters.value_counts().sort_index().head(15))

count    71518.000000
mean         1.422942
std          1.090740
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         40.000000
Name: count, dtype: float64

Encounter frequency:
count
1     54745
2     10434
3      3328
4      1421
5       717
6       346
7       207
8       111
9        70
10       42
11       20
12       19
13       14
14        5
15        9
Name: count, dtype: int64


## 5. Categorical Value Validation

We inspect the unique values of selected categorical features to identify unexpected or invalid categories.

At this stage, no values are modified or removed.

In [6]:
categorical_check = [
    "race",
    "gender",
    "age",
    "weight",
    "change",
    "diabetesMed",
    "insulin",
    "A1Cresult",
    "max_glu_serum",
    "readmitted"
]

for col in categorical_check:
    print(f"\n{'=' * 60}")
    print(f"{col}")
    print(f"{'=' * 60}")
    print(df_clean[col].value_counts(dropna=False).to_string())


race
race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641

gender
gender
Female             54708
Male               47055
Unknown/Invalid        3

age
age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161

weight
weight
?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3

change
change
No    54755
Ch    47011

diabetesMed
diabetesMed
Yes    78363
No     23403

insulin
insulin
No        47383
Steady    30849
Down      12218
Up        11316

A1Cresult
A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812

max_glu_serum
max_glu_serum
NaN     96420
Norm     2597
>200     1485
>300     1264

readmitted
readmitted

## 6. Non-Informative and Identifier Features

We review features that may not represent meaningful predictive information.

Constant features contain no variation and are candidates for removal.

Identifier columns such as `encounter_id` and `patient_nbr` require separate consideration because they identify encounters or patients rather than clinical characteristics.

No feature is removed yet. Final decisions will be made after considering their role in the prediction task and the splitting strategy.

In [7]:
constant_features = [
    col for col in df_clean.columns
    if df_clean[col].nunique(dropna=False) <= 1
]

print("Constant features:")
for col in constant_features:
    print(f"  - {col}")

print("\nIdentifier features:")
for col in ["encounter_id", "patient_nbr"]:
    print(
        f"  - {col}: "
        f"{df_clean[col].nunique():,} unique values "
        f"out of {len(df_clean):,} rows"
    )

Constant features:
  - examide
  - citoglipton

Identifier features:
  - encounter_id: 101,766 unique values out of 101,766 rows
  - patient_nbr: 71,518 unique values out of 101,766 rows


## 7. Standardize Unknown Values for Missingness Analysis

The raw dataset represents unknown categorical values with `?`.

For the purpose of missingness analysis, these values are converted to `NaN`. This allows us to quantify missingness consistently across both categorical and numerical representations.

No rows or features are removed in this step.

In [8]:
df_clean = df_clean.replace("?", np.nan)

missing_summary = (
    df_clean.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df_clean) * 100
).round(2)

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

missing_summary

,missing_count,missing_percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


## 8. Missingness Interpretation

Missing values in this dataset do not necessarily represent random data loss.

Some variables may be missing because a measurement or service was not performed during the encounter. Therefore, missingness is treated as a potentially informative characteristic rather than automatically imputed or removed.

Initial assessment:

| Feature | Missingness | Initial interpretation |
|---|---:|---|
| `weight` | 96.86% | Extremely sparse; limited practical value |
| `max_glu_serum` | 94.75% | Measurement may not have been performed |
| `A1Cresult` | 83.28% | Measurement may not have been performed |
| `medical_specialty` | 49.08% | Missing may represent unavailable/not recorded specialty |
| `payer_code` | 39.56% | Administrative information; substantial missingness |
| `race` | 2.23% | Low missingness |
| `diag_3` | 1.40% | Low missingness |
| `diag_2` | 0.35% | Very low missingness |
| `diag_1` | 0.02% | Very low missingness |

No feature is removed solely because of missingness at this stage.

## 9. Assess Extremely Sparse Features

Features with extremely high missingness are investigated before removal.

We compare the amount of available information and the number of distinct observed values. A feature is not removed solely because it has a high missing rate; its potential predictive and clinical value must also be considered.

In [9]:
sparse_features = ["weight", "max_glu_serum", "A1Cresult"]

sparse_summary = pd.DataFrame({
    "missing_count": df_clean[sparse_features].isna().sum(),
    "missing_percentage": (
        df_clean[sparse_features].isna().mean() * 100
    ).round(2),
    "observed_count": df_clean[sparse_features].notna().sum(),
    "observed_unique_values": df_clean[sparse_features].nunique(dropna=True)
})

sparse_summary

,missing_count,missing_percentage,observed_count,observed_unique_values
weight,98569,96.86,3197,9
max_glu_serum,96420,94.75,5346,3
A1Cresult,84748,83.28,17018,3


## 10. Remove Constant and Extremely Sparse Features

Based on the raw-data assessment:

- `examide` and `citoglipton` are constant and contain no predictive information.
- `weight` has 96.86% missing values and only 3,197 observed records. Its extremely limited coverage makes it unsuitable for the initial modeling pipeline.

Therefore, these features are removed from the working dataset.

`max_glu_serum` and `A1Cresult` are retained because their missingness may represent whether the corresponding measurement was performed. Their missing values will be handled as an explicit category during preprocessing.

In [10]:
features_to_drop = [
    "examide",
    "citoglipton",
    "weight"
]

df_clean = df_clean.drop(columns=features_to_drop)

print("Removed features:")
for feature in features_to_drop:
    print(f"  - {feature}")

print(f"\nNew shape: {df_clean.shape}")

Removed features:
  - examide
  - citoglipton
  - weight

New shape: (101766, 47)


## 11. Validate Rare and Invalid Categorical Values

Categorical variables may contain rare or explicitly invalid values.

We inspect the frequency of rare categories to determine whether they should be retained, grouped, or treated as missing.

No rows are removed automatically.

In [11]:
rare_category_summary = {}

for col in ["race", "gender"]:
    counts = df_clean[col].value_counts(dropna=False)
    rare_category_summary[col] = counts[counts <= 10]

for col, values in rare_category_summary.items():
    print(f"\n{col}")
    print("-" * 40)
    print(values.to_string())


race
----------------------------------------
Series([], )

gender
----------------------------------------
gender
Unknown/Invalid    3


## 12. Handle Invalid Gender Values

The `gender` feature contains only three `Unknown/Invalid` records.

Because these records are extremely rare, they are treated as missing rather than removed. This preserves the observations while allowing the preprocessing pipeline to handle the missing category consistently.

In [12]:
invalid_gender_count = (df_clean["gender"] == "Unknown/Invalid").sum()

df_clean["gender"] = df_clean["gender"].replace(
    "Unknown/Invalid",
    np.nan
)

print(f"Invalid gender values converted to NaN: {invalid_gender_count}")
print(f"Remaining missing gender values: {df_clean['gender'].isna().sum()}")

Invalid gender values converted to NaN: 3
Remaining missing gender values: 3


## 13. Validate Diagnosis Features

The diagnosis variables contain ICD-9 diagnosis codes represented as categorical values.

We inspect their observed formats and missingness after standardizing unknown values to `NaN`.

Diagnosis values will not be numerically interpreted as continuous variables because the codes represent medical categories rather than ordered numerical measurements.


In [13]:
diagnosis_cols = ["diag_1", "diag_2", "diag_3"]

for col in diagnosis_cols:
    print(f"\n{col}")
    print("-" * 40)
    print(f"Missing: {df_clean[col].isna().sum():,}")
    print(f"Unique observed values: {df_clean[col].nunique(dropna=True):,}")
    print("Sample values:")
    print(df_clean[col].dropna().sample(
        n=min(15, df_clean[col].notna().sum()),
        random_state=42
    ).tolist())


diag_1
----------------------------------------
Missing: 21
Unique observed values: 716
Sample values:
['599', '250.7', '414', '577', '428', '434', '511', '784', '413', '410', '428', '532', '414', '428', '411']

diag_2
----------------------------------------
Missing: 358
Unique observed values: 748
Sample values:
['250.43', '250.92', '250', '250.01', '794', '8', '250.02', '211', '250', '427', '250', '493', '574', '162', '574']

diag_3
----------------------------------------
Missing: 1,423
Unique observed values: 789
Sample values:
['199', '491', '599', '414', '272', '518', '599', '590', '428', '562', '998', '250.6', '401', '162', '401']


## 14. Validate Administrative Categorical Features

`payer_code` and `medical_specialty` contain substantial missingness.

Before deciding whether to retain or remove them, we inspect:

- Missing values
- Number of observed categories
- Most frequent categories
- Concentration of observations among the most common categories

No feature is removed based on missingness alone.

In [14]:
administrative_cols = ["payer_code", "medical_specialty"]

for col in administrative_cols:
    counts = df_clean[col].value_counts(dropna=False)

    print(f"\n{'=' * 60}")
    print(col)
    print(f"{'=' * 60}")
    print(f"Missing: {df_clean[col].isna().sum():,}")
    print(f"Observed unique categories: {df_clean[col].nunique(dropna=True):,}")
    print("\nTop categories:")
    print(counts.head(15).to_string())


payer_code
Missing: 40,256
Observed unique categories: 17

Top categories:
payer_code
NaN    40256
MC     32439
HM      6274
SP      5007
BC      4655
MD      3532
CP      2533
UN      2448
CM      1937
OG      1033
PO       592
DM       549
CH       146
WC       135
OT        95

medical_specialty
Missing: 49,949
Observed unique categories: 72

Top categories:
medical_specialty
NaN                                49949
InternalMedicine                   14635
Emergency/Trauma                    7565
Family/GeneralPractice              7440
Cardiology                          5352
Surgery-General                     3099
Nephrology                          1613
Orthopedics                         1400
Orthopedics-Reconstructive          1233
Radiologist                         1140
Pulmonology                          871
Psychiatry                           854
Urology                              685
ObstetricsandGynecology              671
Surgery-Cardiovascular/Thoracic      652


## 15. Cleaning Decisions Summary

The initial cleaning assessment has been completed.

### Removed Features

| Feature | Reason |
|---|---|
| `examide` | Constant feature |
| `citoglipton` | Constant feature |
| `weight` | 96.86% missing with very limited observed information |

### Retained Features with Missing Values

| Feature | Missingness | Decision |
|---|---:|---|
| `max_glu_serum` | 94.75% | Retain; missingness may indicate measurement not performed |
| `A1Cresult` | 83.28% | Retain; missingness may indicate measurement not performed |
| `medical_specialty` | 49.08% | Retain; domain-relevant categorical feature |
| `payer_code` | 39.56% | Retain; potentially informative administrative feature |
| `race` | 2.23% | Retain |
| `diag_3` | 1.40% | Retain |
| `diag_2` | 0.35% | Retain |
| `diag_1` | 0.02% | Retain |
| `gender` | 3 invalid values | Convert `Unknown/Invalid` to missing |

### Important Principle

Missing categorical values will be handled during the preprocessing stage rather than using global imputation before model validation.

No target-based cleaning decision has been performed.

The dataset is now ready for the next stage: defining the prediction target and preparing the train/test split.

In [15]:
print("Final cleaning validation")
print("-" * 50)

print(f"Shape: {df_clean.shape}")
print(f"Exact duplicates: {df_clean.duplicated().sum():,}")
print(f"Missing values: {df_clean.isna().sum().sum():,}")

print("\nRemaining constant features:")
print([
    col for col in df_clean.columns
    if df_clean[col].nunique(dropna=False) <= 1
])

print("\nTarget distribution:")
print(df_clean["readmitted"].value_counts(dropna=False))

Final cleaning validation
--------------------------------------------------
Shape: (101766, 47)
Exact duplicates: 0
Missing values: 275,451

Remaining constant features:
[]

Target distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


## Conclusion

The raw dataset was systematically inspected and cleaned without applying model-specific transformations.

The following actions were completed:

- Standardized unknown values represented by `?` as missing values.
- Removed constant features: `examide` and `citoglipton`.
- Removed `weight` because of its extremely high missingness (96.86%) and limited observed information.
- Converted `Unknown/Invalid` values in `gender` to missing values.
- Retained potentially informative features with high missingness, including `A1Cresult`, `max_glu_serum`, `medical_specialty`, and `payer_code`.
- Confirmed that there are no exact duplicate rows.
- Confirmed that no constant features remain.

No model-specific preprocessing, feature selection, optimization, or train/test split was performed in this notebook.

The cleaned dataset will be saved as an intermediate artifact and used as the starting point for target definition and leakage-resistant train/test splitting.

In [16]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "cleaned_data.csv"

df_clean.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"Shape: {df_clean.shape}")

Cleaned dataset saved to: ..\data\processed\cleaned_data.csv
Shape: (101766, 47)
